# MovieLens 1M content-based recommendation

This notebook transfers the content-based recommender to MovieLens 1M using the exact users, split, candidates, and evaluator from Lijie's model-based workstream.

## Objective

For every user, rank one hidden movie they rated 4 or 5 among 99 fixed unseen distractors.

- Primary metric: NDCG@10
- Secondary metric: Hit Rate@10
- Validation selects M1–M4
- Test is scored only after selection

## Content configurations

| Configuration | Movie representation |
| --- | --- |
| M1 | Genre TF-IDF |
| M2 | Genre TF-IDF + release-decade one-hot |
| M3 | Genre TF-IDF + title TF-IDF |
| M4 | Genre TF-IDF + title TF-IDF + release decade |

Each block is L2-normalized before concatenation. The complete vector is then normalized again. This prevents the large title vocabulary from automatically overwhelming genre or decade.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "data.py").exists():
            return path
    raise FileNotFoundError("Run this notebook inside the dam repository.")

REPO_ROOT = find_repo_root()
PROJECT_DIR = REPO_ROOT / "content_collab" / "movielens_dataset"
for path in (REPO_ROOT, PROJECT_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from movielens_content import (
    assert_archived_counts,
    build_movie_features,
    build_user_item_weights,
    build_user_profiles,
    evaluate_predictions,
    prepare_shared_data,
    resolve_dataset_dir,
    score_candidates,
)
from src.evaluate import rank_candidates

EVAL_USER_LIMIT = (
    int(os.environ["MOVIELENS_CONTENT_USER_LIMIT"])
    if os.getenv("MOVIELENS_CONTENT_USER_LIMIT")
    else None
)

DATASET_DIR = resolve_dataset_dir(
    os.getenv("MOVIELENS_1M_DIR"),
    REPO_ROOT,
    auto_download=True,
)
print("Dataset:", DATASET_DIR)
print("Mode:", "FULL" if EVAL_USER_LIMIT is None else f"SMOKE ({EVAL_USER_LIMIT} users)")

## 1. Recreate and verify the shared protocol

In [ ]:
shared = prepare_shared_data(DATASET_DIR)
assert_archived_counts(shared)

if EVAL_USER_LIMIT is None:
    evaluation_users = shared.eligible_users
else:
    if not 1 <= EVAL_USER_LIMIT <= len(shared.eligible_users):
        raise ValueError("MOVIELENS_CONTENT_USER_LIMIT is outside the user range")
    evaluation_users = shared.eligible_users[:EVAL_USER_LIMIT]

validation_candidates = shared.validation_candidates.loc[
    shared.validation_candidates["user_id"].isin(evaluation_users)
].reset_index(drop=True)
test_candidates = shared.test_candidates.loc[
    shared.test_candidates["user_id"].isin(evaluation_users)
].reset_index(drop=True)

summary = pd.Series({
    "eligible users": len(shared.eligible_users),
    "warm movies": len(shared.warm_items),
    "training positives": len(shared.split.train),
    "validation candidates used": len(validation_candidates),
    "test candidates used": len(test_candidates),
})
display(summary.to_frame("count"))
assert len(validation_candidates) == len(evaluation_users) * 100
assert len(test_candidates) == len(evaluation_users) * 100
print("✅ Shared protocol verified.")

## 2. Build M1–M4 movie vectors

Concrete M4 example:

```text
Toy Story (1995)
→ genre block: Animation, Children's, Comedy
→ title block: toy, story, toy story
→ decade block: 1990s
→ concatenate normalized blocks
→ normalize final vector
```

In [ ]:
features = build_movie_features(shared.metadata)
item_vectors = features.configurations

dimension_table = pd.DataFrame({
    "configuration": list(item_vectors),
    "dimensions": [matrix.shape[1] for matrix in item_vectors.values()],
    "nonzero values": [matrix.nnz for matrix in item_vectors.values()],
})
display(dimension_table)

assert item_vectors["M1"].shape == (3_125, 18)
for name, matrix in item_vectors.items():
    norms = np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1
    assert np.allclose(norms[norms > 0], 1.0, atol=1e-5)
print("✅ Movie dimensions and norms are valid.")

### Verify balanced block influence

For movies with all selected blocks present, normalized concatenation gives equal squared-magnitude shares:

- M2: approximately 50% genre and 50% decade
- M3: approximately 50% genre and 50% title
- M4: approximately one third per block

In [ ]:
def squared_norm(matrix: sparse.csr_matrix) -> np.ndarray:
    return np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel()

block_energy = {
    "genre": squared_norm(features.genre_block),
    "title": squared_norm(features.title_block),
    "decade": squared_norm(features.decade_block),
}
block_rows = []
for configuration, blocks in {
    "M2": ["genre", "decade"],
    "M3": ["genre", "title"],
    "M4": ["genre", "title", "decade"],
}.items():
    total = sum(block_energy[block] for block in blocks)
    for block in blocks:
        share = np.divide(
            block_energy[block],
            total,
            out=np.zeros_like(total),
            where=total > 0,
        )
        block_rows.append({
            "configuration": configuration,
            "block": block,
            "mean squared-magnitude share": share.mean(),
            "median squared-magnitude share": np.median(share),
        })
display(pd.DataFrame(block_rows).round(4))

## 3. Build rating-weighted user profiles

```text
rating 4 → weight 1
rating 5 → weight 2
```

Only positive training interactions contribute.

In [ ]:
user_item_weights, user_to_index, item_to_index = build_user_item_weights(shared)
assert set(np.unique(user_item_weights.data)) == {1.0, 2.0}

def profiles_for(name: str) -> sparse.csr_matrix:
    return build_user_profiles(user_item_weights, item_vectors[name])

example_user = evaluation_users[0]
example_history = shared.split.train.loc[
    shared.split.train["user_id"].eq(example_user)
].merge(
    shared.metadata[["movie_id", "title_with_year"]],
    left_on="anime_id",
    right_on="movie_id",
    how="left",
)
example_history["profile_weight"] = example_history["rating"] - 3
display(
    example_history[
        ["title_with_year", "rating", "profile_weight"]
    ].sort_values("rating", ascending=False).head(15)
)

## 4. Check the popularity baseline

The full validation result must reproduce the archived MovieLens popularity baseline before content configurations are evaluated.

In [ ]:
popularity = shared.split.train.groupby("anime_id")["user_id"].nunique()
popularity_predictions = validation_candidates.copy()
popularity_predictions["score"] = (
    popularity_predictions["item_id"].map(popularity).fillna(0).astype(float)
)
popularity_validation, _ = evaluate_predictions(popularity_predictions)
display(pd.Series(popularity_validation).to_frame("validation"))

if EVAL_USER_LIMIT is None:
    assert np.isclose(
        popularity_validation["ndcg@10"],
        0.36625509043761767,
    )
    assert np.isclose(
        popularity_validation["hit_rate@10"],
        0.5971163407358303,
    )
    print("✅ Archived validation popularity metrics reproduced.")
else:
    print("Subset diagnostic only; full parity assertions are skipped.")

## 5. Select M1–M4 on validation only

In [ ]:
validation_rows = []
validation_user_ranks = {}

for name, movie_vectors in item_vectors.items():
    print(f"Evaluating {name} ...")
    profiles = profiles_for(name)
    predictions = score_candidates(
        validation_candidates,
        profiles,
        movie_vectors,
        user_to_index,
        item_to_index,
    )
    metrics, user_metrics = evaluate_predictions(predictions)
    validation_rows.append({"configuration": name, **metrics})
    validation_user_ranks[name] = user_metrics[["user_id", "rank"]].rename(
        columns={"rank": f"{name}_rank"}
    )
    del profiles, predictions

validation_results = (
    pd.DataFrame(validation_rows)
    .sort_values(["ndcg@10", "configuration"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_configuration = str(validation_results.loc[0, "configuration"])

display(validation_results.round({"ndcg@10": 6, "hit_rate@10": 6}))
print("Selected by validation NDCG@10:", selected_configuration)

## 6. Evaluate the selected configuration on test once

This is the first model-scoring call on the fixed test candidates.

In [ ]:
selected_item_vectors = item_vectors[selected_configuration]
selected_profiles = profiles_for(selected_configuration)
test_predictions = score_candidates(
    test_candidates,
    selected_profiles,
    selected_item_vectors,
    user_to_index,
    item_to_index,
)
test_metrics, test_user_metrics = evaluate_predictions(test_predictions)

content_test_result = pd.DataFrame([{
    "model": f"Content {selected_configuration}",
    "ndcg@10": test_metrics["ndcg@10"],
    "hit_rate@10": test_metrics["hit_rate@10"],
    "users": test_metrics["user_count"],
    "mode": "full" if EVAL_USER_LIMIT is None else f"smoke:{EVAL_USER_LIMIT}",
}])
display(content_test_result.round({"ndcg@10": 6, "hit_rate@10": 6}))

## 7. Compare with Lijie's MovieLens results

Only the full run belongs in this comparison table.

In [ ]:
reference_results = pd.DataFrame([
    {"model": "Popularity", "ndcg@10": 0.3555955818, "hit_rate@10": 0.5807093139},
    {"model": "GMF", "ndcg@10": 0.507165, "hit_rate@10": 0.758314},
    {"model": "NeuMF", "ndcg@10": 0.599754, "hit_rate@10": 0.825710},
    {"model": "BPR", "ndcg@10": 0.620558, "hit_rate@10": 0.825820},
    {"model": "Weighted NeuMF", "ndcg@10": 0.623203, "hit_rate@10": 0.831013},
    {"model": "BPR + Weighted NeuMF", "ndcg@10": 0.632039, "hit_rate@10": 0.838692},
])
comparison = pd.concat(
    [
        reference_results,
        content_test_result[["model", "ndcg@10", "hit_rate@10"]],
    ],
    ignore_index=True,
)
display(
    comparison.sort_values("ndcg@10", ascending=False)
    .reset_index(drop=True)
    .round(6)
)

## 8. Success and failure cases

The case analysis shows the positive movie's rank and the training movies most similar to it.

In [ ]:
ranked_test = rank_candidates(test_predictions)
positive_test = ranked_test.loc[
    ranked_test["label"].eq(1),
    ["user_id", "item_id", "rank"],
]

def choose_user(frame: pd.DataFrame, largest: bool = False):
    if frame.empty:
        return None
    ordered = frame.sort_values("rank", ascending=not largest)
    return int(ordered.iloc[0]["user_id"])

case_users = {
    "rank-1 success": choose_user(positive_test.loc[positive_test["rank"].eq(1)]),
    "top-10 boundary": choose_user(
        positive_test.loc[positive_test["rank"].le(10)],
        largest=True,
    ),
    "failure outside top 10": choose_user(
        positive_test.loc[positive_test["rank"].gt(10)]
    ),
}
display(pd.Series(case_users, name="user_id").to_frame())

In [ ]:
def explain_user(user_id: int, top_n: int = 10):
    user_ranked = ranked_test.loc[
        ranked_test["user_id"].eq(user_id)
    ].head(top_n).merge(
        shared.metadata[
            ["movie_id", "title_with_year", "genres", "release_decade"]
        ],
        left_on="item_id",
        right_on="movie_id",
        how="left",
    )

    positive_item = int(
        ranked_test.loc[
            ranked_test["user_id"].eq(user_id)
            & ranked_test["label"].eq(1),
            "item_id",
        ].iloc[0]
    )
    target_index = item_to_index[positive_item]
    history = shared.split.train.loc[
        shared.split.train["user_id"].eq(user_id)
    ].copy()
    history_indices = history["anime_id"].map(item_to_index).to_numpy()
    similarity = np.asarray(
        selected_item_vectors[history_indices]
        .multiply(selected_item_vectors[target_index])
        .sum(axis=1)
    ).ravel()
    history["contribution"] = (history["rating"].to_numpy() - 3) * similarity
    contributors = history.nlargest(5, "contribution").merge(
        shared.metadata[
            ["movie_id", "title_with_year", "genres", "release_decade"]
        ],
        left_on="anime_id",
        right_on="movie_id",
        how="left",
    )
    return user_ranked, contributors

for label, user_id in case_users.items():
    if user_id is None:
        print(f"No {label} in this evaluation subset.")
        continue
    print(f"\n{label.upper()} | user {user_id}")
    top, contributors = explain_user(user_id)
    display(
        top[
            [
                "rank",
                "label",
                "score",
                "title_with_year",
                "genres",
                "release_decade",
            ]
        ]
    )
    print("Strongest training contributors to the held-out movie:")
    display(
        contributors[
            [
                "rating",
                "contribution",
                "title_with_year",
                "genres",
                "release_decade",
            ]
        ]
    )

## 9. Configuration rank changes

This shows users for whom title or decade information most changes the held-out validation rank.

In [ ]:
rank_comparison = None
for frame in validation_user_ranks.values():
    rank_comparison = (
        frame
        if rank_comparison is None
        else rank_comparison.merge(frame, on="user_id")
    )

rank_comparison["M4_minus_M1_improvement"] = (
    rank_comparison["M1_rank"] - rank_comparison["M4_rank"]
)
display(
    rank_comparison.nlargest(15, "M4_minus_M1_improvement")
)

## Evaluation limits

This experiment measures retrieval of a known positive among 99 unobserved distractors.

It does not measure:

- exact rating prediction
- chronological next-watch prediction
- confirmed dislike detection
- full-catalog ranking

MovieLens content results should be compared with Anime by model behaviour and relative gains, not by assuming that raw metric differences measure dataset difficulty.